# Predicting Customer Term Deposit Subscription Using Supervised Machine Learning

## 1. Project Overview

### 1.1 Background

### 1.2 Problem Statement

### 1.3 Project Objective

### 1.4 Analytical Questions

In [1]:
import sys
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Project root
PROJECT_ROOT = Path.cwd().parent

# Allow imports from src/
sys.path.append(str(PROJECT_ROOT))

# Import project modules


# Dataset
DATA_PATH = PROJECT_ROOT / "data" / "raw" / "bank-full.csv"

df = pd.read_csv(DATA_PATH, sep=";")

print(f"Dataset loaded successfully: {df.shape}")

Dataset loaded successfully: (45211, 17)


In [2]:
from src.data_inspection import DataInspection

In [3]:
inspection = DataInspection(df)
inspection.inspect(target="y")

DATASET OVERVIEW
Rows: 45,211
Columns: 17

Column Names:
['age', 'job', 'marital', 'education', 'default', 'balance', 'housing', 'loan', 'contact', 'day', 'month', 'duration', 'campaign', 'pdays', 'previous', 'poutcome', 'y']

DATA TYPES
age           int64
job          object
marital      object
education    object
default      object
balance       int64
housing      object
loan         object
contact      object
day           int64
month        object
duration      int64
campaign      int64
pdays         int64
previous      int64
poutcome     object
y            object
dtype: object

MISSING VALUES
No standard missing values detected.

PLACEHOLDER VALUES: 'unknown'
      column  count  percentage
3   poutcome  36959       81.75
2    contact  13020       28.80
1  education   1857        4.11
0        job    288        0.64

DUPLICATE RECORDS
Duplicate rows: 0

NUMERICAL SUMMARY
            count         mean          std     min    25%    50%     75%  \
age       45211.0    40.936210 

In [4]:
unknown_summary = inspection.placeholder_values()

unknown_summary


PLACEHOLDER VALUES: 'unknown'
      column  count  percentage
3   poutcome  36959       81.75
2    contact  13020       28.80
1  education   1857        4.11
0        job    288        0.64


,column,count,percentage
3,poutcome,36959,81.75
2,contact,13020,28.80
1,education,1857,4.11
0,job,288,0.64


In [5]:
duplicate_count = df.duplicated().sum()

print(f"Duplicate rows: {duplicate_count:,}")

Duplicate rows: 0


In [6]:
missing_summary = df.isnull().sum()

print(missing_summary[missing_summary > 0])

Series([], dtype: int64)


In [7]:
# Separate features and target

X = df.drop(columns=["y", "duration"])
y = df["y"]

print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)

print("\nFeatures:")
print(X.columns.tolist())

print("\nTarget distribution:")
print(y.value_counts())

Feature matrix shape: (45211, 15)
Target shape: (45211,)

Features:
['age', 'job', 'marital', 'education', 'default', 'balance', 'housing', 'loan', 'contact', 'day', 'month', 'campaign', 'pdays', 'previous', 'poutcome']

Target distribution:
y
no     39922
yes     5289
Name: count, dtype: int64


In [8]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training set:")
print(f"X_train: {X_train.shape}")
print(f"y_train: {y_train.shape}")

print("\nTest set:")
print(f"X_test: {X_test.shape}")
print(f"y_test: {y_test.shape}")

Training set:
X_train: (36168, 15)
y_train: (36168,)

Test set:
X_test: (9043, 15)
y_test: (9043,)


In [9]:
print("Training target distribution:")
print(
    y_train.value_counts(normalize=True)
    .mul(100)
    .round(2)
)

print("\nTest target distribution:")
print(
    y_test.value_counts(normalize=True)
    .mul(100)
    .round(2)
)

Training target distribution:
y
no     88.3
yes    11.7
Name: proportion, dtype: float64

Test target distribution:
y
no     88.3
yes    11.7
Name: proportion, dtype: float64


In [ ]:
numerical_features = X_train.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X_train.select_dtypes(
    include=["object"]
).columns.tolist()

print("Numerical features:")
print(numerical_features)

print("\nCategorical features:")
print(categorical_features)

Numerical features:
['age', 'balance', 'day', 'campaign', 'pdays', 'previous']

Categorical features:
['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'poutcome']


In [11]:
from src.preprocessing import DataPreprocessor

preprocessor_builder = DataPreprocessor(
    numerical_features=numerical_features,
    categorical_features=categorical_features
)

preprocessor = preprocessor_builder.build_pipeline()

In [12]:
# Test the preprocessing pipeline on the training data

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Original training shape:", X_train.shape)
print("Processed training shape:", X_train_processed.shape)

print("Original test shape:", X_test.shape)
print("Processed test shape:", X_test_processed.shape)

print("\nProcessed data type:", type(X_train_processed))

Original training shape: (36168, 15)
Processed training shape: (36168, 50)
Original test shape: (9043, 15)
Processed test shape: (9043, 50)

Processed data type: <class 'numpy.ndarray'>


In [13]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.pipeline import Pipeline


# Encode target
target_mapping = {"no": 0, "yes": 1}

y_train_encoded = y_train.map(target_mapping)
y_test_encoded = y_test.map(target_mapping)

print("Encoded target distribution:")
print(y_train_encoded.value_counts())

Encoded target distribution:
y
0    31937
1     4231
Name: count, dtype: int64


In [14]:
# Create baseline model pipelines

logistic_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            LogisticRegression(
                max_iter=1000,
                random_state=42
            )
        )
    ]
)


decision_tree_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            DecisionTreeClassifier(
                random_state=42
            )
        )
    ]
)


random_forest_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            RandomForestClassifier(
                n_estimators=200,
                random_state=42,
                n_jobs=-1
            )
        )
    ]
)

print("Three baseline classification pipelines created successfully.")

Three baseline classification pipelines created successfully.


In [15]:
import time

models = {
    "Logistic Regression": logistic_model,
    "Decision Tree": decision_tree_model,
    "Random Forest": random_forest_model
}

training_times = {}

for name, model in models.items():

    print(f"\nTraining {name}...")

    start_time = time.time()

    model.fit(X_train, y_train_encoded)

    elapsed_time = time.time() - start_time

    training_times[name] = elapsed_time

    print(f"{name} trained successfully.")
    print(f"Training time: {elapsed_time:.2f} seconds")


Training Logistic Regression...
Logistic Regression trained successfully.
Training time: 2.19 seconds

Training Decision Tree...
Decision Tree trained successfully.
Training time: 5.01 seconds

Training Random Forest...
Random Forest trained successfully.
Training time: 22.73 seconds


In [16]:
from src.evaluate_models import ModelEvaluator

evaluator = ModelEvaluator(models)

In [17]:
baseline_results = evaluator.evaluate(
    X_test,
    y_test_encoded
)

baseline_results

,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,Logistic Regression,0.893288,0.663158,0.178639,0.281459,0.771749
1,Decision Tree,0.830919,0.298201,0.328922,0.312809,0.613178
2,Random Forest,0.893730,0.622166,0.233459,0.339519,0.792895


In [18]:
confusion_matrices = evaluator.confusion_matrices(
    X_test,
    y_test_encoded
)

for name, matrix in confusion_matrices.items():

    print(f"\n{name}")
    print("-" * 40)
    print(matrix)


Logistic Regression
----------------------------------------
[[7889   96]
 [ 869  189]]

Decision Tree
----------------------------------------
[[7166  819]
 [ 710  348]]

Random Forest
----------------------------------------
[[7835  150]
 [ 811  247]]


In [19]:
# Class-weighted model pipelines

logistic_weighted = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            LogisticRegression(
                class_weight="balanced",
                max_iter=1000,
                random_state=42
            )
        )
    ]
)


decision_tree_weighted = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            DecisionTreeClassifier(
                class_weight="balanced",
                random_state=42
            )
        )
    ]
)


random_forest_weighted = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            RandomForestClassifier(
                n_estimators=200,
                class_weight="balanced",
                random_state=42,
                n_jobs=-1
            )
        )
    ]
)

weighted_models = {
    "Logistic Regression (Balanced)": logistic_weighted,
    "Decision Tree (Balanced)": decision_tree_weighted,
    "Random Forest (Balanced)": random_forest_weighted
}

print("Class-weighted model pipelines created successfully.")

Class-weighted model pipelines created successfully.


In [20]:
for name, model in weighted_models.items():

    print(f"\nTraining {name}...")

    start_time = time.time()

    model.fit(
        X_train,
        y_train_encoded
    )

    elapsed_time = time.time() - start_time

    print(f"{name} trained successfully.")
    print(f"Training time: {elapsed_time:.2f} seconds")


Training Logistic Regression (Balanced)...
Logistic Regression (Balanced) trained successfully.
Training time: 5.60 seconds

Training Decision Tree (Balanced)...
Decision Tree (Balanced) trained successfully.
Training time: 4.72 seconds

Training Random Forest (Balanced)...
Random Forest (Balanced) trained successfully.
Training time: 16.23 seconds


In [21]:
weighted_evaluator = ModelEvaluator(weighted_models)

weighted_results = weighted_evaluator.evaluate(
    X_test,
    y_test_encoded
)

weighted_results

,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,Logistic Regression (Balanced),0.754838,0.266236,0.623819,0.373198,0.772238
1,Decision Tree (Balanced),0.840097,0.308300,0.294896,0.301449,0.603616
2,Random Forest (Balanced),0.880681,0.488268,0.413043,0.447517,0.791260


In [22]:
all_results = pd.concat(
    [
        baseline_results,
        weighted_results
    ],
    ignore_index=True
)

all_results

,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,Logistic Regression,0.893288,0.663158,0.178639,0.281459,0.771749
1,Decision Tree,0.830919,0.298201,0.328922,0.312809,0.613178
2,Random Forest,0.893730,0.622166,0.233459,0.339519,0.792895
3,Logistic Regression (Balanced),0.754838,0.266236,0.623819,0.373198,0.772238
4,Decision Tree (Balanced),0.840097,0.308300,0.294896,0.301449,0.603616
5,Random Forest (Balanced),0.880681,0.488268,0.413043,0.447517,0.791260


In [ ]:
from imblearn.over_sampling import SMOTENC
from imblearn.pipeline import Pipeline as ImbPipeline

print("imbalanced-learn is available.")